In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets accelerate evaluate scikit-learn matplotlib seaborn

In [ ]:
from google.colab import files
print("Hãy upload 2 file: vihallu-train.csv và vihallu-test.csv")
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score
import torch

In [ ]:
train_df = pd.read_csv("vihallu-train.csv")
test_df = pd.read_csv("vihallu-test.csv")

In [ ]:
print("Dữ liệu huấn luyện:")
display(train_df.head())
print(f"Số dòng train: {len(train_df)}, test: {len(test_df)}")

In [ ]:
print("\nPhân bố nhãn:")
print(train_df['label'].value_counts())
plt.figure(figsize=(6,4))
sns.countplot(data=train_df, x="label", palette="Set2")
plt.title("Phân bố nhãn trong tập huấn luyện")
plt.show()

train_df["context_len"] = train_df["context"].apply(lambda x: len(str(x).split()))
train_df["prompt_len"] = train_df["prompt"].apply(lambda x: len(str(x).split()))
train_df["respone_len"] = train_df["respone"].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(8,4))
sns.boxplot(data=train_df[["context_len","prompt_len","respone_len"]])
plt.title("Phân phối độ dài các trường văn bản")
plt.show()

In [ ]:
train_df = train_df.dropna(subset=["context", "prompt", "respone", "label"]).reset_index(drop=True)

label2id = {"no": 0, "intrinsic": 1, "extrinsic": 2}
id2label = {v: k for k, v in label2id.items()}
train_df["label_id"] = train_df["label"].map(label2id)

train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df["label_id"], random_state=42)
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}")

In [ ]:
MODEL_NAME = "VietAI/vit5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    text = [
        f"Context: {c}\nPrompt: {p}\nResponse: {r}"
        for c, p, r in zip(examples["context"], examples["prompt"], examples["respone"])
    ]
    return tokenizer(text, truncation=True, padding="max_length", max_length=512)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

dataset = DatasetDict({
    "train": train_dataset.map(preprocess, batched=True),
    "validation": val_dataset.map(preprocess, batched=True)
})

dataset = dataset.remove_columns(["context", "prompt", "respone", "label", "__index_level_0__"])
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label_id"])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./vit5-hallucination",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3e-5,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_dir="./logs",
    logging_steps=50,
    metric_for_best_model="f1",
    save_total_limit=2
)

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
preds = trainer.predict(dataset["validation"])
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=label2id.keys()))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(label2id.keys()))
disp.plot(cmap="Blues")
plt.show()

In [ ]:
SAVE_DIR = "./vit5-hallucination/best_model"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model và tokenizer đã lưu tại: {SAVE_DIR}")

import shutil
shutil.make_archive("vit5-hallucination", 'zip', SAVE_DIR)
print("Đã tạo file vit5-hallucination.zip — có thể tải về từ Colab Files tab.")